Humans communicate in natural language, but machines require structured commands.

Your task is to design a model that translates commands like:

> turn on the kitchen light

into

> INTENT=LIGHT_ON ROOM=KITCHEN.

# Input:
A single English sentence describing a command.
Example:

```
turn off the bedroom light
set temperature to 22 degrees
turn on the kitchen light

```

# Output:
A sequence of tokens representing the structured command.
Examples:



```
INTENT=LIGHT_OFF ROOM=BEDROOM
INTENT=SET_TEMP VALUE=22
INTENT=LIGHT_ON ROOM=KITCHEN

```

# Steps:
1. Build a dataset.
2. Build a model (including embedding layer, encoder and decoder)
3. Train
4. Test








In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# =========================
# 1. دیتاست
# =========================
data = [
    ("turn off the bedroom light", "LIGHT_OFF BEDROOM"),
    ("turn on the kitchen light", "LIGHT_ON KITCHEN"),
    ("set temperature to 22 degrees", "SET_TEMP 22"),
    ("set temperature to 18 degrees", "SET_TEMP 18"),
    ("turn on the bathroom light", "LIGHT_ON BATHROOM"),
    ("turn off the living room light", "LIGHT_OFF LIVING_ROOM"),
    ("set temperature to 25 degrees", "SET_TEMP 25"),
    ("turn on the bedroom light", "LIGHT_ON BEDROOM")
]

inputs, targets = zip(*data)
targets = ["<start> " + t + " <end>" for t in targets]

# =========================
# 2. توکن‌سازی
# =========================
tokenizer_in = Tokenizer()
tokenizer_in.fit_on_texts(inputs)

tokenizer_out = Tokenizer(filters='')
tokenizer_out.fit_on_texts(targets)

X = tokenizer_in.texts_to_sequences(inputs)
Y = tokenizer_out.texts_to_sequences(targets)

max_l_in = max(len(x) for x in X)
max_l_out = max(len(y) for y in Y)

X_pad = pad_sequences(X, maxlen=max_l_in, padding='post')
Y_pad = pad_sequences(Y, maxlen=max_l_out, padding='post')

num_encoder_tokens = len(tokenizer_in.word_index) + 1
num_decoder_tokens = len(tokenizer_out.word_index) + 1

# =========================
# 3. مدل Seq2Seq با LSTM
# =========================
emb_dim = 64
latent_dim = 128

# --- Encoder ---
encoder_inputs = layers.Input(shape=(max_l_in,))
encoder_emb = layers.Embedding(num_encoder_tokens, emb_dim)(encoder_inputs)
_, state_h, state_c = layers.LSTM(latent_dim, return_state=True)(encoder_emb)
encoder_states = [state_h, state_c]

# --- Decoder ---
decoder_inputs = layers.Input(shape=(max_l_out,))
decoder_embedding_layer = layers.Embedding(num_decoder_tokens, emb_dim) # Define the embedding layer
decoder_emb = decoder_embedding_layer(decoder_inputs) # Get its output
decoder_lstm = layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_emb, initial_state=encoder_states)
decoder_dense = layers.Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# --- مدل آموزش ---
model = models.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# =========================
# 4. آماده سازی Y برای آموزش
# =========================
# خروجی decoder باید با یک توکن شیفت داده شود
decoder_target_data = np.zeros_like(Y_pad)
decoder_target_data[:, :-1] = Y_pad[:, 1:]
decoder_target_data[:, -1] = 0  # padding آخر

# =========================
# 5. آموزش مدل
# =========================
print("Starting training...")
model.fit([X_pad, Y_pad], decoder_target_data, epochs=300, verbose=0)
print("Training finished!")

# =========================
# 6. مدل‌های Encoder و Decoder برای inference
# =========================
# Encoder
encoder_model = models.Model(encoder_inputs, encoder_states)

# Decoder
decoder_state_input_h = layers.Input(shape=(latent_dim,))
decoder_state_input_c = layers.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_emb2 = decoder_embedding_layer(decoder_inputs) # Reuse the embedding layer for inference
decoder_outputs2, state_h2, state_c2 = decoder_lstm(decoder_emb2, initial_state=decoder_states_inputs)
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_states2 = [state_h2, state_c2]

decoder_model = models.Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs2] + decoder_states2)

# =========================
# 7. توکن‌های معکوس
# =========================
reverse_target_index = {v: k for k, v in tokenizer_out.word_index.items()}

# =========================
# 8. تابع decode_sequence
# =========================
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = tokenizer_out.word_index['<start>']

    stop_condition = False
    decoded_sentence = ''
    h, c = states_value

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + [h, c])
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_index.get(sampled_token_index, '')

        if sampled_word == '<end>' or len(decoded_sentence.split()) > max_l_out:
            stop_condition = True
        else:
            decoded_sentence += sampled_word + ' '
            target_seq = np.zeros((1, 1))
            target_seq[0, 0] = sampled_token_index

    return decoded_sentence.strip()

# =========================
# 9. تست مدل
# =========================
test_sentences = [
    "turn off the bedroom light",
    "turn on the kitchen light",
    "set temperature to 25 degrees",
    "turn on the bathroom light"
]

for sent in test_sentences:
    seq = tokenizer_in.texts_to_sequences([sent])
    seq_pad = pad_sequences(seq, maxlen=max_l_in, padding='post')
    decoded = decode_sequence(seq_pad)
    print(f"Input: {sent}")
    print(f"Predicted structured command: {decoded}\n")

Starting training...
Training finished!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Input: turn off the bedroom light
Predicted structured command: light_off bedroom

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Input: turn on the kitchen light
Predicted structured command: light_on kitchen

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Input: set temperature to 25 degrees
Predicted structured command: set_temp 25

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Input: turn on the bathroom light
Predicted structured command: light_on bathroom

